<a href="https://colab.research.google.com/github/evildead23151/SentinelGov/blob/main/notebookd5416d58ba.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(42)

N_TRANSACTIONS = 50_000
N_VENDORS = 500
N_DEPARTMENTS = 25

# IDs
df = pd.DataFrame({
    "transaction_id": range(1, N_TRANSACTIONS + 1),
    "vendor_id": np.random.choice(
        [f"V_{i}" for i in range(N_VENDORS)],
        size=N_TRANSACTIONS,
        p=np.random.dirichlet(np.ones(N_VENDORS))  # vendor imbalance
    ),
    "department_id": np.random.choice(
        [f"D_{i}" for i in range(N_DEPARTMENTS)],
        size=N_TRANSACTIONS
    ),
    "transaction_date": pd.to_datetime("2023-01-01") +
        pd.to_timedelta(np.random.randint(0, 365, size=N_TRANSACTIONS), unit="D"),
})

# Amounts (heavy-tailed, realistic)
df["amount"] = (
    np.random.lognormal(mean=8.5, sigma=1.1, size=N_TRANSACTIONS)
).round(2)

# Invoice IDs
df["invoice_id"] = [
    f"INV_{np.random.randint(100000, 999999)}"
    for _ in range(N_TRANSACTIONS)
]

df = df.sort_values("transaction_date").reset_index(drop=True)

df.head()


,transaction_id,vendor_id,department_id,transaction_date,amount,invoice_id
0,46350,V_323,D_5,2023-01-01,7896.23,INV_504704
1,25885,V_113,D_23,2023-01-01,1458.03,INV_828196
2,21539,V_204,D_20,2023-01-01,2508.83,INV_957990
3,1127,V_112,D_9,2023-01-01,6314.17,INV_661126
4,42279,V_321,D_7,2023-01-01,7751.81,INV_696882


In [ ]:
# ===============================
# SentinelGov Synthetic Dataset
# ===============================

import pandas as pd
import numpy as np

# -------------------------------
# Global config
# -------------------------------
np.random.seed(42)

N_TRANSACTIONS = 50_000
N_VENDORS = 500
N_DEPARTMENTS = 25

# -------------------------------
# STEP 1: Base synthetic data
# -------------------------------
df = pd.DataFrame({
    "transaction_id": range(1, N_TRANSACTIONS + 1),
    "vendor_id": np.random.choice(
        [f"V_{i}" for i in range(N_VENDORS)],
        size=N_TRANSACTIONS,
        p=np.random.dirichlet(np.ones(N_VENDORS))
    ),
    "department_id": np.random.choice(
        [f"D_{i}" for i in range(N_DEPARTMENTS)],
        size=N_TRANSACTIONS
    ),
    "transaction_date": pd.to_datetime("2023-01-01") +
        pd.to_timedelta(
            np.random.randint(0, 365, size=N_TRANSACTIONS),
            unit="D"
        ),
})

# Heavy-tailed realistic amounts
df["amount"] = (
    np.random.lognormal(mean=8.5, sigma=1.1, size=N_TRANSACTIONS)
).round(2)

# Random invoice IDs
df["invoice_id"] = [
    f"INV_{np.random.randint(100000, 999999)}"
    for _ in range(N_TRANSACTIONS)
]

# Fraud metadata
df["is_synthetic_fraud"] = 0
df["fraud_type"] = None

df = df.sort_values("transaction_date").reset_index(drop=True)

# -------------------------------
# STEP 2: Inject fraud patterns
# -------------------------------

# ---- 1. Duplicate invoices ----
dup_indices = np.random.choice(df.index, size=300, replace=False)

df.loc[dup_indices, "invoice_id"] = "DUPLICATE_INV_001"
df.loc[dup_indices, "is_synthetic_fraud"] = 1
df.loc[dup_indices, "fraud_type"] = "DUPLICATE_INVOICE"

# ---- 2. Structuring / smurfing ----
struct_rows = pd.DataFrame({
    "transaction_id": range(1_000_001, 1_000_051),
    "vendor_id": "V_999",
    "department_id": "D_1",
    "transaction_date": pd.date_range(
        start="2023-06-01",
        periods=50,
        freq="D"
    ),
    "amount": np.random.uniform(9000, 9900, size=50),
    "invoice_id": [f"STR_{i}" for i in range(50)],
    "is_synthetic_fraud": 1,
    "fraud_type": "STRUCTURING"
})

df = pd.concat([df, struct_rows], ignore_index=True)

# ---- 3. Vendor hopping across departments ----
hop_rows = []
hop_vendor = "V_888"

for dept in ["D_3", "D_7", "D_12", "D_18"]:
    for i in range(5):
        hop_rows.append({
            "transaction_id": np.random.randint(2_000_000, 3_000_000),
            "vendor_id": hop_vendor,
            "department_id": dept,
            "transaction_date": pd.to_datetime("2023-09-01") + pd.Timedelta(days=i),
            "amount": np.random.uniform(40000, 60000),
            "invoice_id": f"HOP_{dept}_{i}",
            "is_synthetic_fraud": 1,
            "fraud_type": "VENDOR_HOPPING"
        })

hop_df = pd.DataFrame(hop_rows)
df = pd.concat([df, hop_df], ignore_index=True)

# -------------------------------
# Final cleanup
# -------------------------------
df = df.sort_values("transaction_date").reset_index(drop=True)

# -------------------------------
# Sanity checks
# -------------------------------
print("Total rows:", len(df))
print("\nFraud type distribution:")
print(df["fraud_type"].value_counts(dropna=False))

df.head()


Total rows: 50070

Fraud type distribution:
fraud_type
None                 49700
DUPLICATE_INVOICE      300
STRUCTURING             50
VENDOR_HOPPING          20
Name: count, dtype: int64


,transaction_id,vendor_id,department_id,transaction_date,amount,invoice_id,is_synthetic_fraud,fraud_type
0,46350,V_323,D_5,2023-01-01,7896.23,INV_504704,0,None
1,28937,V_343,D_3,2023-01-01,2582.01,INV_223311,0,None
2,38698,V_73,D_18,2023-01-01,10220.20,INV_462869,0,None
3,39351,V_386,D_16,2023-01-01,23984.98,INV_810114,0,None
4,47080,V_403,D_13,2023-01-01,5525.23,INV_306910,0,None


In [ ]:
# ==========================================================
# SentinelGov — Clean End-to-End Feature Pipeline (ONE CELL)
# ==========================================================

import numpy as np
import pandas as pd

np.random.seed(42)

# -------------------------------
# 1. Generate synthetic data
# -------------------------------
N_TRANSACTIONS = 50_000
N_VENDORS = 500
N_DEPARTMENTS = 25

df = pd.DataFrame({
    "transaction_id": range(1, N_TRANSACTIONS + 1),
    "vendor_id": np.random.choice(
        [f"V_{i}" for i in range(N_VENDORS)],
        size=N_TRANSACTIONS,
        p=np.random.dirichlet(np.ones(N_VENDORS))
    ),
    "department_id": np.random.choice(
        [f"D_{i}" for i in range(N_DEPARTMENTS)],
        size=N_TRANSACTIONS
    ),
    "transaction_date": pd.to_datetime("2023-01-01") +
        pd.to_timedelta(np.random.randint(0, 365, size=N_TRANSACTIONS), unit="D"),
    "amount": np.random.lognormal(mean=8.5, sigma=1.1, size=N_TRANSACTIONS).round(2),
    "invoice_id": [f"INV_{np.random.randint(100000,999999)}" for _ in range(N_TRANSACTIONS)]
})

df["is_synthetic_fraud"] = 0
df["fraud_type"] = None

df = df.sort_values("transaction_date").reset_index(drop=True)

# -------------------------------
# 2. Inject fraud
# -------------------------------
# Duplicate invoices
dup_idx = np.random.choice(df.index, 300, replace=False)
df.loc[dup_idx, "invoice_id"] = "DUPLICATE_INV"
df.loc[dup_idx, ["is_synthetic_fraud","fraud_type"]] = [1,"DUPLICATE_INVOICE"]

# Structuring
struct = pd.DataFrame({
    "transaction_id": range(1_000_001, 1_000_051),
    "vendor_id": "V_999",
    "department_id": "D_1",
    "transaction_date": pd.date_range("2023-06-01", periods=50),
    "amount": np.random.uniform(9000, 9900, 50),
    "invoice_id": [f"STR_{i}" for i in range(50)],
    "is_synthetic_fraud": 1,
    "fraud_type": "STRUCTURING"
})
df = pd.concat([df, struct], ignore_index=True)

# Vendor hopping
rows = []
for dept in ["D_3","D_7","D_12","D_18"]:
    for i in range(5):
        rows.append({
            "transaction_id": np.random.randint(2_000_000,3_000_000),
            "vendor_id": "V_888",
            "department_id": dept,
            "transaction_date": pd.to_datetime("2023-09-01")+pd.Timedelta(days=i),
            "amount": np.random.uniform(40000,60000),
            "invoice_id": f"HOP_{dept}_{i}",
            "is_synthetic_fraud": 1,
            "fraud_type": "VENDOR_HOPPING"
        })
df = pd.concat([df, pd.DataFrame(rows)], ignore_index=True)

df = df.sort_values("transaction_date").reset_index(drop=True)

# -------------------------------
# 3. Feature engineering
# -------------------------------
df["amount_log"] = np.log1p(df["amount"])

def robust_z(s):
    m = s.median()
    mad = np.median(np.abs(s - m)) + 1e-6
    return (s - m) / mad

df["amount_z_vendor"] = (
    df.groupby("vendor_id")["amount"]
      .transform(robust_z)
      .clip(-10,10)
)

df["department_code"] = df["department_id"].astype("category").cat.codes

def time_feats(g):
    g = g.sort_values("transaction_date")
    g["vendor_txn_count_7d"] = g.rolling("7D", on="transaction_date")["transaction_id"].count()
    g["vendor_txn_count_30d"] = g.rolling("30D", on="transaction_date")["transaction_id"].count()
    g["unique_departments_30d"] = (
        g.rolling("30D", on="transaction_date")["department_code"]
        .apply(lambda x: len(set(x)), raw=True)
    )
    return g

df = df.groupby("vendor_id", group_keys=False).apply(time_feats, include_groups=False)


ml_features = [
    "amount_log",
    "amount_z_vendor",
    "vendor_txn_count_7d",
    "vendor_txn_count_30d",
    "unique_departments_30d"
]

df[ml_features] = df[ml_features].fillna(0)

# -------------------------------
# 4. Sanity check
# -------------------------------
print("Rows:", len(df))
print(df["fraud_type"].value_counts(dropna=False))
df[ml_features].head()


Rows: 50070
fraud_type
None                 49700
DUPLICATE_INVOICE      300
STRUCTURING             50
VENDOR_HOPPING          20
Name: count, dtype: int64


,amount_log,amount_z_vendor,vendor_txn_count_7d,vendor_txn_count_30d,unique_departments_30d
651,9.724335,1.823971,1.0,1.0,1.0
1156,8.653062,-0.314697,2.0,2.0,2.0
2892,8.888159,-0.019342,1.0,3.0,3.0
2997,8.382358,-0.578990,2.0,4.0,4.0
5031,9.209865,0.515637,1.0,4.0,4.0


In [ ]:
# ==========================================
# STEP 4: Isolation Forest (Audit-Safe)
# ==========================================

from sklearn.ensemble import IsolationForest
import numpy as np

# ------------------------------------------
# Feature matrix for ML
# ------------------------------------------
ml_features = [
    "amount_log",
    "amount_z_vendor",
    "vendor_txn_count_7d",
    "vendor_txn_count_30d",
    "unique_departments_30d"
]

X = df[ml_features].copy()

# ------------------------------------------
# Train only on assumed-normal data
# (exclude known synthetic fraud)
# ------------------------------------------
train_mask = df["is_synthetic_fraud"] == 0
X_train = X.loc[train_mask]

# ------------------------------------------
# Isolation Forest configuration
# ------------------------------------------
iso = IsolationForest(
    n_estimators=200,
    contamination=0.03,   # conservative assumption
    max_samples="auto",
    random_state=42,
    n_jobs=-1
)

iso.fit(X_train)

# ------------------------------------------
# Raw anomaly score
# (higher = more anomalous)
# ------------------------------------------
df["ml_raw_score"] = -iso.score_samples(X)

# ------------------------------------------
# Percentile normalization (0–100)
# ------------------------------------------
df["ml_anomaly_score"] = (
    df["ml_raw_score"]
    .rank(pct=True)
    .mul(100)
    .round(2)
)

# ------------------------------------------
# Sanity checks
# ------------------------------------------
print("ML anomaly score distribution:")
print(df["ml_anomaly_score"].describe())

print("\nFraud vs Normal (mean ML score):")
print(
    df.groupby("fraud_type", dropna=False)["ml_anomaly_score"]
      .mean()
      .round(2)
)


ML anomaly score distribution:
count    50070.000000
mean        50.000998
std         28.867800
min          0.000000
25%         25.000000
50%         50.000000
75%         75.000000
max        100.000000
Name: ml_anomaly_score, dtype: float64

Fraud vs Normal (mean ML score):
fraud_type
DUPLICATE_INVOICE    47.48
STRUCTURING          89.67
VENDOR_HOPPING       95.31
NaN                  49.96
Name: ml_anomaly_score, dtype: float64


In [ ]:
# ==========================================
# STEP 5: Risk Scoring & Explainability
# ==========================================

import numpy as np

# ------------------------------------------
# 1. Rule severity definition
# ------------------------------------------
RULE_SEVERITY = {
    "DUPLICATE_INVOICE": 90,
    "STRUCTURING": 80,
    "VENDOR_HOPPING": 70
}

# ------------------------------------------
# 2. Rule score (dominant signal)
# ------------------------------------------
def compute_rule_score(row):
    if row["fraud_type"] in RULE_SEVERITY:
        return RULE_SEVERITY[row["fraud_type"]]
    return 0

df["rule_score"] = df.apply(compute_rule_score, axis=1)

# ------------------------------------------
# 3. Statistical deviation score (lightweight)
# ------------------------------------------
df["stat_score"] = (
    df["amount_z_vendor"]
    .abs()
    .clip(0, 5)
    .mul(20)      # scale to 0–100
)

# ------------------------------------------
# 4. Final risk score (transparent formula)
# ------------------------------------------
df["final_risk_score"] = (
    0.65 * df["rule_score"] +
    0.20 * df["ml_anomaly_score"] +
    0.15 * df["stat_score"]
).round(2)

# ------------------------------------------
# 5. Risk banding (for UI / audit triage)
# ------------------------------------------
def risk_band(score):
    if score >= 75:
        return "HIGH"
    elif score >= 40:
        return "MEDIUM"
    return "LOW"

df["risk_band"] = df["final_risk_score"].apply(risk_band)

# ------------------------------------------
# 6. Explainability layer
# ------------------------------------------
def generate_explanation(row):
    reasons = []

    # Rule-based explanation
    if row["rule_score"] > 0:
        reasons.append(
            f"Matched predefined audit rule: {row['fraud_type']}."
        )

    # ML-based explanation
    if row["ml_anomaly_score"] >= 85:
        reasons.append(
            "Transaction exhibits an unusual combination of vendor behavior patterns "
            "compared to historical norms."
        )

    # Statistical explanation
    if abs(row["amount_z_vendor"]) >= 3:
        reasons.append(
            "Transaction amount deviates significantly from this vendor’s typical range."
        )

    if not reasons:
        reasons.append(
            "Transaction shows minor deviations but no direct rule violations."
        )

    return " ".join(reasons) + " This alert indicates elevated review priority, not wrongdoing."

df["explanation"] = df.apply(generate_explanation, axis=1)

# ------------------------------------------
# Sanity checks
# ------------------------------------------
print("Final risk score distribution:")
print(df["final_risk_score"].describe())

print("\nRisk band counts:")
print(df["risk_band"].value_counts())

print("\nSample HIGH risk alerts:")
df[df["risk_band"] == "HIGH"][
    [
        "fraud_type",
        "rule_score",
        "ml_anomaly_score",
        "stat_score",
        "final_risk_score",
        "explanation"
    ]
].head(5)


Final risk score distribution:
count    50070.000000
mean        14.938326
std         10.362939
min          0.010000
25%          7.310000
50%         13.320000
75%         20.100000
max         93.460000
Name: final_risk_score, dtype: float64

Risk band counts:
risk_band
LOW       49700
MEDIUM      265
HIGH        105
Name: count, dtype: int64

Sample HIGH risk alerts:


,fraud_type,rule_score,ml_anomaly_score,stat_score,final_risk_score,explanation
10149,DUPLICATE_INVOICE,90,73.71,94.003789,87.34,Matched predefined audit rule: DUPLICATE_INVOI...
17377,DUPLICATE_INVOICE,90,79.62,13.727488,76.48,Matched predefined audit rule: DUPLICATE_INVOI...
39035,DUPLICATE_INVOICE,90,97.93,100.000000,93.09,Matched predefined audit rule: DUPLICATE_INVOI...
4932,DUPLICATE_INVOICE,90,95.31,100.000000,92.56,Matched predefined audit rule: DUPLICATE_INVOI...
39377,DUPLICATE_INVOICE,90,91.54,67.728300,86.97,Matched predefined audit rule: DUPLICATE_INVOI...


In [ ]:
# ==========================================================
# STEP 5: Risk Scoring, Banding & Explainability (FINAL)
# ==========================================================

import numpy as np
import pandas as pd

# ----------------------------------------------------------
# 1. Rule severity configuration (deterministic, dominant)
# ----------------------------------------------------------
RULE_SEVERITY = {
    "DUPLICATE_INVOICE": 90,
    "STRUCTURING": 80,
    "VENDOR_HOPPING": 70
}

# ----------------------------------------------------------
# 2. Rule score
# ----------------------------------------------------------
df["rule_score"] = df["fraud_type"].map(RULE_SEVERITY).fillna(0)

# ----------------------------------------------------------
# 3. Statistical deviation score (contextual, lightweight)
# ----------------------------------------------------------
df["stat_score"] = (
    df["amount_z_vendor"]
    .abs()
    .clip(0, 5)
    .mul(20)        # scale to 0–100
)

# ----------------------------------------------------------
# 4. Final risk score (transparent formula)
# Rules dominate, ML refines confidence
# ----------------------------------------------------------
df["final_risk_score"] = (
    0.65 * df["rule_score"] +
    0.20 * df["ml_anomaly_score"] +
    0.15 * df["stat_score"]
).clip(0, 100).round(2)

# ----------------------------------------------------------
# 5. Risk banding (audit triage)
# ----------------------------------------------------------
def risk_band(score):
    if score >= 75:
        return "HIGH"
    elif score >= 40:
        return "MEDIUM"
    return "LOW"

df["risk_band"] = df["final_risk_score"].apply(risk_band)

# ----------------------------------------------------------
# 6. Primary trigger (for UI / judge clarity)
# ----------------------------------------------------------
def primary_trigger(row):
    if row["rule_score"] > 0:
        return "RULE"
    if row["ml_anomaly_score"] >= 85:
        return "ML"
    if row["stat_score"] >= 60:
        return "STATISTICAL"
    return "LOW_SIGNAL"

df["primary_trigger"] = df.apply(primary_trigger, axis=1)

# ----------------------------------------------------------
# 7. Explainability (auditor-safe language)
# ----------------------------------------------------------
def generate_explanation(row):
    reasons = []

    if row["rule_score"] > 0:
        reasons.append(
            f"Matched predefined audit rule: {row['fraud_type']}."
        )

    if row["ml_anomaly_score"] >= 85:
        reasons.append(
            "Transaction exhibits an unusual combination of vendor behavior "
            "compared to historical norms."
        )

    if abs(row["amount_z_vendor"]) >= 3:
        reasons.append(
            "Transaction amount deviates significantly from this vendor’s "
            "typical transaction range."
        )

    if not reasons:
        reasons.append(
            "Transaction shows minor deviations but no direct rule violations."
        )

    return (
        " ".join(reasons)
        + " This alert indicates elevated review priority and does not imply wrongdoing."
    )

df["explanation"] = df.apply(generate_explanation, axis=1)

# ----------------------------------------------------------
# 8. Final sanity outputs (judge-friendly)
# ----------------------------------------------------------
print("\nFinal risk score distribution:")
print(df["final_risk_score"].describe())

print("\nRisk band counts:")
print(df["risk_band"].value_counts())

print("\nFraud vs normal — mean final risk score:")
print(
    df.groupby("fraud_type", dropna=False)["final_risk_score"]
      .mean()
      .round(2)
)

print("\nSample HIGH risk alerts:")
df[df["risk_band"] == "HIGH"][
    [
        "fraud_type",
        "rule_score",
        "ml_anomaly_score",
        "stat_score",
        "final_risk_score",
        "primary_trigger",
        "explanation"
    ]
].head(5)



Final risk score distribution:
count    50070.000000
mean        14.938326
std         10.362939
min          0.010000
25%          7.310000
50%         13.320000
75%         20.100000
max         93.460000
Name: final_risk_score, dtype: float64

Risk band counts:
risk_band
LOW       49700
MEDIUM      265
HIGH        105
Name: count, dtype: int64

Fraud vs normal — mean final risk score:
fraud_type
DUPLICATE_INVOICE    72.29
STRUCTURING          72.93
VENDOR_HOPPING       67.87
NaN                  14.51
Name: final_risk_score, dtype: float64

Sample HIGH risk alerts:


,fraud_type,rule_score,ml_anomaly_score,stat_score,final_risk_score,primary_trigger,explanation
10149,DUPLICATE_INVOICE,90.0,73.71,94.003789,87.34,RULE,Matched predefined audit rule: DUPLICATE_INVOI...
17377,DUPLICATE_INVOICE,90.0,79.62,13.727488,76.48,RULE,Matched predefined audit rule: DUPLICATE_INVOI...
39035,DUPLICATE_INVOICE,90.0,97.93,100.000000,93.09,RULE,Matched predefined audit rule: DUPLICATE_INVOI...
4932,DUPLICATE_INVOICE,90.0,95.31,100.000000,92.56,RULE,Matched predefined audit rule: DUPLICATE_INVOI...
39377,DUPLICATE_INVOICE,90.0,91.54,67.728300,86.97,RULE,Matched predefined audit rule: DUPLICATE_INVOI...


In [ ]:
# ==========================================================
# STEP 6: Package Risk Model as Pickle (Anti-Gravity Ready)
# ==========================================================

import pickle
from datetime import datetime

# ----------------------------------------------------------
# Model metadata
# ----------------------------------------------------------
MODEL_BUNDLE = {
    "model_name": "SentinelGov Risk Scoring Model",
    "model_version": "iforest_v1.0",
    "created_at": datetime.utcnow().isoformat() + "Z",
    "description": (
        "Isolation Forest–based anomaly scoring model used as a "
        "supporting signal for procurement risk triage. "
        "Model does not determine fraud."
    )
}

# ----------------------------------------------------------
# Feature schema (ORDER MATTERS)
# ----------------------------------------------------------
FEATURE_SCHEMA = [
    "amount_log",
    "amount_z_vendor",
    "vendor_txn_count_7d",
    "vendor_txn_count_30d",
    "unique_departments_30d"
]

# ----------------------------------------------------------
# Rule severity configuration
# ----------------------------------------------------------
RULE_SEVERITY = {
    "DUPLICATE_INVOICE": 90,
    "STRUCTURING": 80,
    "VENDOR_HOPPING": 70
}

# ----------------------------------------------------------
# Scoring weights (transparent & configurable)
# ----------------------------------------------------------
SCORING_WEIGHTS = {
    "rule_weight": 0.65,
    "ml_weight": 0.20,
    "stat_weight": 0.15
}

# ----------------------------------------------------------
# Assemble model bundle
# ----------------------------------------------------------
risk_model_bundle = {
    "metadata": MODEL_BUNDLE,
    "isolation_forest_model": iso,
    "feature_schema": FEATURE_SCHEMA,
    "rule_severity": RULE_SEVERITY,
    "scoring_weights": SCORING_WEIGHTS
}

# ----------------------------------------------------------
# Save pickle
# ----------------------------------------------------------
with open("sentinelgov_risk_model_v1.pkl", "wb") as f:
    pickle.dump(risk_model_bundle, f)

print("✅ Model pickle created: sentinelgov_risk_model_v1.pkl")
print("Included:")
print("- Isolation Forest model")
print("- Feature schema")
print("- Rule severity config")
print("- Scoring weights")
print("- Version metadata")


✅ Model pickle created: sentinelgov_risk_model_v1.pkl
Included:
- Isolation Forest model
- Feature schema
- Rule severity config
- Scoring weights
- Version metadata


/tmp/ipykernel_55/1373202236.py:14: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "created_at": datetime.utcnow().isoformat() + "Z",
